# Module 4 — Signal processing libraries

**Who this is for:** you can handle arrays and want to know which library does what, and how to use it safely.

> **Self-check — skip this module if you can already:**
>
> - [ ] get a correct frequency axis for an FFT and a Welch PSD of a signal sampled at `fs`
> - [ ] design a band-pass filter as SOS and apply it with zero phase
> - [ ] say when to use `filtfilt` and when a causal filter is required
> - [ ] find peaks with `find_peaks` and turn them into heart rate
>
> If you can tick every box, go to the next module. If not, work through the parts you can't tick. Before
> jumping to a section, run the module's **setup cell** (the first code cell): later sections use variables
> defined earlier, so run the cells above a section if you get a `NameError`.

**You will learn**
- which library to reach for (overview table)
- FFT and the frequency axis
- PSD with Welch and the spectrogram
- filter design and zero-phase filtering with SOS
- notch filters, resampling and detrending
- peaks and heart rate
- optional: envelopes and wavelets
- a checklist for safe signal processing

**Where the course uses this:** Chapters 3–10 and Labs 3–10; `preprocess()` and `extract_features()` in every capstone track.

*Run each cell with **Shift+Enter**, top to bottom. Nothing here needs data files; once the libraries are
installed (see the [overview](README.md#setup-first)), the notebook runs offline.*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/04_signal_processing_libraries.ipynb)
&nbsp; Part of the **Python for biosignal processing** tutorial — see the [overview](README.md) to choose which modules you need.

## 0. Which library for what

| Task | Library | Typical functions |
|---|---|---|
| Arrays, FFT | `numpy` | `np.fft.rfft`, `np.fft.rfftfreq` |
| Filters, spectra, peaks, resampling | `scipy.signal` | `butter`, `sosfiltfilt`, `iirnotch`, `welch`, `spectrogram`, `find_peaks`, `resample_poly`, `hilbert`, `detrend` |
| Wavelets | `pywt` (PyWavelets) | `wavedec`, `waverec`, `cwt` |
| EEG/MEG/PSG reading, montages, epochs | `mne` | `read_raw_edf`, `Raw.filter`, `Epochs` |
| ECG / PhysioNet records | `wfdb` | `rdrecord`, `rdann` |
| Machine learning | `scikit-learn` | see Module 5 |
| Plotting | `matplotlib` | `plt.subplots` |

Other libraries you may meet: `neurokit2` (ready-made ECG/EDA/RSP pipelines), `statsmodels` (AR models,
statistics), `librosa` (audio, spectral features). They are fine to use in the capstone **if you can explain
what they do** — a black-box call you cannot justify costs marks under *design justification*.

In [ ]:
# Colab only: install PyWavelets if this runtime lacks it (locally it comes with requirements-lock.txt).
import sys, importlib.util
if "google.colab" in sys.modules and importlib.util.find_spec("pywt") is None:
    %pip install -q PyWavelets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.integrate import trapezoid      # area under a curve (band power)
import pywt

rng = np.random.default_rng(2013)
fs = 250
t = np.arange(10 * fs) / fs
clean = 1.0 * np.sin(2 * np.pi * 1.2 * t) + 0.5 * np.sin(2 * np.pi * 10 * t)   # the signal we care about
mains = 0.3 * np.sin(2 * np.pi * 50 * t)                                          # 50 Hz interference
noise = 0.2 * rng.standard_normal(t.size)                                         # broadband noise
x = clean + mains + noise

## 1. FFT and the frequency axis

For real signals use `rfft` (positive frequencies only) and **always build the frequency axis with
`rfftfreq(n, d=1/fs)`**. Getting this axis wrong is the most common FFT mistake.

To read an **amplitude**, scale by 1/N and double the positive-frequency bins, because each one stands for a
pair of positive and negative frequencies. The two exceptions are **DC** (bin 0) and, when N is even, the
**Nyquist** bin: they have no partner, so they are not doubled. Test the function on signals whose answer
you know.

In [ ]:
def amplitude_spectrum(x, fs):
    """Single-sided amplitude spectrum: returns (frequencies in Hz, amplitudes in the units of x)."""
    a = np.abs(np.fft.rfft(x)) / x.size
    if x.size % 2 == 0:
        a[1:-1] *= 2           # even N: the last bin is Nyquist, which is not doubled
    else:
        a[1:] *= 2             # odd N: there is no Nyquist bin
    return np.fft.rfftfreq(x.size, d=1 / fs), a

# known answers: a constant 3, and a 0.5-amplitude sinusoid exactly on a frequency bin
f_test, a_test = amplitude_spectrum(np.full(1000, 3.0), fs)
assert np.isclose(a_test[0], 3.0)
f_test, a_test = amplitude_spectrum(0.5 * np.sin(2 * np.pi * 10 * t), fs)
assert np.isclose(a_test[np.argmin(abs(f_test - 10))], 0.5)

f, amp = amplitude_spectrum(x, fs)
print(f"bin spacing = fs/N = {f[1]:.3f} Hz; highest bin = {f[-1]:.1f} Hz (equal to fs/2 only when N is even)")

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(f, amp); ax.set_xlim(0, 60)
ax.set_xlabel("frequency (Hz)"); ax.set_ylabel("amplitude (a.u.)"); ax.set_title("FFT amplitude spectrum")
plt.show()

The amplitudes come out exact here because 1.2, 10 and 50 Hz fall exactly on frequency bins of this
10-s record. A sinusoid **between** bins leaks into its neighbours and its peak looks smaller (Chapter 4):

In [ ]:
for f0 in (10.0, 10.05):
    ff, aa = amplitude_spectrum(0.5 * np.sin(2 * np.pi * f0 * t), fs)
    print(f"true amplitude 0.5 at {f0} Hz -> tallest bin {aa.max():.3f}")

## 2. PSD with Welch, and the spectrogram

A raw FFT of a noisy signal gives a very noisy *estimate* of the spectrum (Chapter 7). **Welch** averages
the spectra of overlapping, windowed segments: the estimate varies much less, but its frequency resolution is
coarser. `nperseg` sets that trade-off. The bin spacing (`fs / nperseg`) is not the same as the resolution,
which is also widened by the window (a Hann window by default). The PSD has units of **signal² per Hz**;
the area under it in a band is the **band power**.

A **spectrogram** shows how the spectrum changes over time (Chapter 3).

In [ ]:
f_w, P = signal.welch(x, fs=fs, nperseg=2 * fs)          # 2-s segments -> 0.5 Hz bin spacing

def band_power(f, P, lo, hi):
    """Area under the PSD from lo to hi Hz. Includes both edges, which is correct when lo and hi are
    on the frequency grid (here every 0.5 Hz). For other limits, interpolate P at lo and hi first."""
    band = (f >= lo) & (f <= hi)
    return trapezoid(P[band], f[band])

# known answer: a flat PSD of 1 per Hz over 4-8 Hz has power 4, and adjacent bands add up
flat = np.ones_like(f_w)
assert np.isclose(band_power(f_w, flat, 4, 8), 4.0)
assert np.isclose(band_power(f_w, flat, 4, 6) + band_power(f_w, flat, 6, 8), band_power(f_w, flat, 4, 8))

print(f"power in 8-12 Hz: {band_power(f_w, P, 8, 12):.3f} a.u.²  (a 0.5-amplitude sinusoid has power 0.125)")

f_s, t_s, S = signal.spectrogram(x, fs=fs, nperseg=fs, noverlap=fs // 2)
fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].semilogy(f_w, P); ax[0].set_xlim(0, 60)
ax[0].set_xlabel("frequency (Hz)"); ax[0].set_ylabel("PSD (a.u.²/Hz)"); ax[0].set_title("Welch PSD")
im = ax[1].pcolormesh(t_s, f_s, 10 * np.log10(S + 1e-12), shading="auto"); ax[1].set_ylim(0, 60)
ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("frequency (Hz)"); ax[1].set_title("spectrogram")
fig.colorbar(im, ax=ax[1], label="PSD (dB re 1 a.u.²/Hz)")
plt.tight_layout(); plt.show()

`signal.spectrogram` is simple and fine for this course. Newer SciPy versions call it *legacy* and offer
`signal.ShortTimeFFT` as the modern interface; you don't need it here. Likewise, `signal.sosfreqz` (used below)
is called `signal.freqz_sos` from SciPy 1.15 on; the old name still works and keeps the notebook compatible with
older SciPy versions.

## 3. Filters: design as SOS, apply with zero phase

- Pass `fs=fs` and give cut-offs **in hertz**; don't hand-normalise.
- Use `output="sos"` (second-order sections): numerically stable at higher orders. Note that
  `butter(4, [lo, hi], btype="bandpass")` gives an 8th-order band-pass (order 4 per edge).
- **Look at the frequency response** before trusting a filter.

### 3a. One pass vs forward–backward: the magnitude

`sosfilt` runs the filter once. `sosfiltfilt` runs it forwards and then backwards, so the signal passes
through it **twice**: the effective response is |H|², twice as steep, and −3 dB at the cut-off becomes
−6 dB.

In [ ]:
sos_bp = signal.butter(4, [0.5, 40], btype="bandpass", fs=fs, output="sos")

w_hz, h = signal.sosfreqz(sos_bp, worN=4096, fs=fs)
gain_db = 20 * np.log10(np.abs(h) + 1e-12)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(w_hz, gain_db, label="one pass: sosfilt, |H|")
ax.plot(w_hz, 2 * gain_db, "--", label="forward-backward: sosfiltfilt, |H|²")
ax.set(xlim=(0, 80), ylim=(-60, 5), xlabel="frequency (Hz)", ylabel="gain (dB)", title="0.5-40 Hz band-pass")
ax.legend(fontsize=8); plt.show()

k40 = np.argmin(abs(w_hz - 40))
print(f"at 40 Hz: one pass {gain_db[k40]:.1f} dB, forward-backward {2 * gain_db[k40]:.1f} dB")

### 3b. One pass vs forward–backward: the timing

Now the **same** band-pass, applied both ways to a single 10 Hz sinusoid. One pass delays it; forward–backward
has **zero phase response**, so a sinusoid in the pass-band keeps its timing. We look at the middle of the
record, away from the edges, where every filter has start-up transients.

In [ ]:
s10 = np.sin(2 * np.pi * 10 * t)
y_causal = signal.sosfilt(sos_bp, s10)          # causal: only past samples, delayed
y_zero = signal.sosfiltfilt(sos_bp, s10)        # zero phase: uses future samples too, offline only

sl = slice(int(5.0 * fs), int(5.3 * fs))       # 0.3 s from the middle of the record
fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(t[sl], s10[sl], "k", lw=2, label="input (10 Hz)")
ax.plot(t[sl], y_zero[sl], "--", label="sosfiltfilt: in step")
ax.plot(t[sl], y_causal[sl], label="sosfilt: delayed")
ax.set(xlabel="time (s)", ylabel="amplitude (a.u.)"); ax.legend(fontsize=8); plt.show()

_, h10 = signal.sosfreqz(sos_bp, worN=[10.0], fs=fs)
print(f"one-pass phase delay at 10 Hz: {-np.angle(h10[0]) / (2 * np.pi * 10) * 1000:.1f} ms")

Use `sosfiltfilt` for offline analysis. Use the causal `sosfilt` when only the past may be used (real-time
systems, or anything you will deploy), and account for its delay.

"Zero phase" does **not** mean every peak of a real signal stays put: a mixture's shape changes when the filter
changes how much of each frequency remains. What it guarantees is that each pass-band frequency is not shifted
in time.

### 3c. Notch filters, detrending and resampling

In [ ]:
sos_notch = signal.tf2sos(*signal.iirnotch(50, Q=30, fs=fs))   # narrow 50 Hz notch (60 Hz in the US)
x_dt = signal.detrend(x)                                  # remove a straight-line trend
x_100 = signal.resample_poly(x, up=2, down=5)             # 250 Hz -> 100 Hz, with anti-alias filtering
x_med = signal.medfilt(x, kernel_size=5)                  # median filter: removes spikes, keeps edges
print(x.size, "->", x_100.size, "samples")

**Never downsample by just taking every n-th sample** (`x[::5]`) unless the signal has already been
low-pass filtered below the new Nyquist frequency: otherwise anything above it folds back as aliasing
(Chapter 2). `resample_poly` and `decimate` do the filtering for you.

## 4. Peaks and heart rate

The pulses below are a toy stand-in for R-peaks: a clean demonstration of `find_peaks`, not a validated ECG
detector. On real ECG, check signal quality and compare against annotated beats before trusting the count.

In [ ]:
ecg_like = np.zeros(t.size)
beat_times = np.arange(0.5, 10, 0.8)                     # a beat every 0.8 s = 75 bpm
for bt in beat_times:
    ecg_like += np.exp(-((t - bt) ** 2) / (2 * 0.01 ** 2))
ecg_like += 0.05 * rng.standard_normal(t.size)

peaks, _ = signal.find_peaks(ecg_like, height=0.5, distance=int(0.3 * fs))   # at least 0.3 s apart
rr = np.diff(peaks) / fs
print(f"{peaks.size} beats found ({beat_times.size} true); "
      f"heart rate = 60 / mean(RR) = {60 / rr.mean():.1f} bpm")

This is heart rate as *beats per minute over the record*, 60 / mean(RR). Module 1 shows why the average of
beat-by-beat rates, mean(60 / RR), can differ.

## 5. Optional extensions: envelopes and wavelets

You can skip this section on a first read; Chapter 5 and Lab 5 cover it in depth.

**Hilbert envelope.** The magnitude of the analytic signal is defined for any real signal, but it only
behaves like the slowly varying amplitude of *one* component when the signal is narrowband. On a broadband
mixture it may not represent the envelope of the component you want, so band-pass first (Lab 5).

In [ ]:
burst = np.sin(2 * np.pi * 40 * t) * np.exp(-((t - 5) ** 2) / 0.5)
envelope = np.abs(signal.hilbert(burst))
print(f"envelope peaks at t = {t[np.argmax(envelope)]:.2f} s (true centre: 5.00 s)")

**Wavelet denoising.** Decompose into scales, shrink the small detail coefficients, reconstruct. Because
we built `x` ourselves, we can judge the result against the **known noise-free signal**, and the result is
a useful warning: applied to every scale, the standard recipe makes things *worse*.

In [ ]:
coeffs = pywt.wavedec(x, "db4", level=5)                  # [approximation, details coarse ... fine]
sigma = np.median(np.abs(coeffs[-1])) / 0.6745            # noise estimate from the finest scale
thr = sigma * np.sqrt(2 * np.log(x.size))                 # the "universal" threshold
reference = clean + mains           # the target: the signal BEFORE broadband noise was added
rms = lambda e: np.sqrt(np.mean(e ** 2))

def denoise(levels):
    """Soft-threshold the finest `levels` detail scales; keep the others unchanged."""
    den = list(coeffs)
    for i in range(len(coeffs) - levels, len(coeffs)):
        den[i] = pywt.threshold(coeffs[i], thr, mode="soft")
    return pywt.waverec(den, "db4")[: x.size]

print(f"error vs noise-free signal, before:           {rms(x - reference):.3f}")
print(f"after thresholding all 5 detail scales:       {rms(denoise(5) - reference):.3f}  <- worse")
print(f"after thresholding only the finest scale:     {rms(denoise(1) - reference):.3f}  <- better")

Why? At 250 Hz the detail scales cover roughly 62–125, 31–62, 16–31, 8–16 and 4–8 Hz. Only the finest
one (62–125 Hz) holds nothing but noise here; the 50 Hz and 10 Hz components live in the others, and
thresholding them shrinks real signal. (The target here is "the signal before broadband noise was added",
which still contains the 50 Hz mains. Removing mains is a separate job for the notch filter; what counts as an
error always depends on the target you choose.) The lesson is the same as for filters: **know where your signal
lives before removing anything**, and check the result against something you trust. On real data, where there is
no noise-free reference, judge a denoiser by what it does to the features you care about.

## 6. Checklist for safe signal processing

1. **Look at the raw signal first.** Know what the noise is before choosing a filter (Chapter 8).
2. **Pass `fs` everywhere** (`butter(..., fs=fs)`, `welch(..., fs=fs)`), and label axes in Hz and seconds.
3. **Test on signals with a known answer:** a constant, a sinusoid on a bin, a flat PSD.
4. **Plot the filter's frequency response** — the one you actually apply, once or twice — and check it keeps
   the band you care about.
5. **Prefer SOS and zero-phase** offline; use causal filters only when you must, and note the delay.
6. **Filter long recordings, not tiny epochs:** filtering short pieces creates edge artefacts.
7. **Check that what matters survives:** compare a feature before and after (Lab 9 does this for delta).
8. **Anything learned from data** (thresholds, ICA, CSP, scalers) must be fitted on training data only (Module 5).

## Try it yourself

Remove the 50 Hz component from `x` with the notch filter `sos_notch` (zero phase). Using a Welch PSD before
and after, show that the 50 Hz peak dropped while the 10 Hz peak stayed: plot both PSDs on one axis, and print
the change in dB at 50 Hz and at 10 Hz.

In [ ]:
# TODO

**Solution:** try it yourself first, then compare with the [solutions notebook](solutions.ipynb) (Module 4). Yours may differ and still be right.

**Next:** Module 5 — turning signals into features and features into honest predictions.